# ML-07 — Baseline Action Score and Top-20 Review

**Lane:** Refresh / Content Opportunity Scoring

**Objective:** Validate useful signals, build a simple baseline scoring rule, rank content opportunities, and review the top 20 recommendations.

This notebook uses only existing content and performance information. It does not use future-window or outcome-derived labels as model inputs.

## 1. Load dataset and inspect columns

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
display(df.head())

## 2. Identify required columns

The following mappings are candidate column names. Verify them against the dataset data dictionary before using the results.

In [ ]:
def find_column(possible_names, columns):
    lower_columns = {col.lower(): col for col in columns}
    for name in possible_names:
        if name.lower() in lower_columns:
            return lower_columns[name.lower()]
    return None

content_col = find_column(
    ["content_id", "url_id", "page_id", "item_id"], df.columns
)

staleness_col = find_column(
    ["days_since_last_update", "days_since_update", "content_age_days", "days_old", "staleness_days"],
    df.columns
)

impressions_col = find_column(
    ["impressions_90d", "impressions", "total_impressions"], df.columns
)

clicks_col = find_column(
    ["clicks_90d", "clicks", "total_clicks"], df.columns
)

ctr_col = find_column(
    ["ctr_90d", "ctr", "click_through_rate"], df.columns
)

position_col = find_column(
    ["avg_position_90d", "avg_position", "average_position", "position"],
    df.columns
)

print("Content ID:", content_col)
print("Staleness:", staleness_col)
print("Impressions:", impressions_col)
print("Clicks:", clicks_col)
print("CTR:", ctr_col)
print("Position:", position_col)

required_columns = [staleness_col, impressions_col, position_col]
if any(col is None for col in required_columns):
    raise ValueError("A required column was not found. Check the dataset schema and update the column mappings.")

if content_col is None:
    df["baseline_content_id"] = np.arange(1, len(df) + 1)
    content_col = "baseline_content_id"

if ctr_col is None and clicks_col is not None:
    df["baseline_ctr"] = np.where(
        pd.to_numeric(df[impressions_col], errors="coerce") > 0,
        pd.to_numeric(df[clicks_col], errors="coerce") /
        pd.to_numeric(df[impressions_col], errors="coerce"),
        np.nan
    )
    ctr_col = "baseline_ctr"

if ctr_col is None:
    raise ValueError("CTR or clicks data is required for the baseline scoring rule.")

for col in [staleness_col, impressions_col, ctr_col, position_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values in selected columns:")
display(df[[content_col, staleness_col, impressions_col, ctr_col, position_col]].isna().sum())

## 3. Prepare signal buckets

The buckets below are baseline analysis groups. Confirm that their thresholds are appropriate for the dataset and assignment definitions.

In [ ]:
analysis_df = df.copy()

analysis_df["staleness_bucket"] = pd.cut(
    analysis_df[staleness_col],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
)

analysis_df["position_bucket"] = pd.cut(
    analysis_df[position_col],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"]
)

print("Staleness bucket counts:")
display(analysis_df["staleness_bucket"].value_counts(dropna=False).sort_index())

print("Position bucket counts:")
display(analysis_df["position_bucket"].value_counts(dropna=False).sort_index())

## 4. Signal 1 — Staleness versus refresh flag

Check whether older content is more likely to be marked for refresh. The flag must be verified against the FlyRank data dictionary before interpreting this as product-flag validation.

In [ ]:
flag_candidates = [
    col for col in analysis_df.columns
    if any(term in col.lower() for term in [
        "refresh_flag", "is_refresh", "stale_flag", "refresh", "quick_win", "ctr_fix"
    ])
]

print("Possible flag columns:", flag_candidates)

if flag_candidates:
    flag_col = flag_candidates[0]
    print("Selected candidate flag:", flag_col)
    print("Verify this column's meaning in the data dictionary before interpreting it.")
else:
    flag_col = None
    print("No candidate product flag found. A proxy analysis will be used instead.")

if flag_col is not None:
    signal_1_df = analysis_df[[staleness_col, flag_col, "staleness_bucket"]].copy()
    signal_1_df[flag_col] = pd.to_numeric(signal_1_df[flag_col], errors="coerce")
    signal_1_table = signal_1_df.groupby("staleness_bucket", observed=False).agg(
        n=(flag_col, "count"),
        refresh_rate=(flag_col, "mean")
    ).reset_index()
    signal_1_table["refresh_rate"] = signal_1_table["refresh_rate"] * 100
    signal_1_table["verdict"] = "MIXED"
    valid_rates = signal_1_table.dropna(subset=["refresh_rate"])
    if len(valid_rates) >= 2:
        difference = valid_rates["refresh_rate"].iloc[-1] - valid_rates["refresh_rate"].iloc[0]
        if difference >= 10:
            verdict_1 = "CONFIRMED"
        elif difference <= -10:
            verdict_1 = "OPPOSITE"
        else:
            verdict_1 = "MIXED"
    else:
        verdict_1 = "FALSE"
    signal_1_table["verdict"] = verdict_1
    signal_1_table["explanation"] = "Refresh-flag rate by staleness bucket; inspect the trend and sample sizes."
else:
    signal_1_table = analysis_df.groupby("staleness_bucket", observed=False).agg(
        n=(ctr_col, "count"),
        median_ctr=(ctr_col, "median")
    ).reset_index()
    signal_1_table["verdict"] = "MIXED"
    signal_1_table["explanation"] = "Proxy only: median CTR by staleness bucket; this does not validate the product refresh flag."

display(signal_1_table)

## 5. Signal 2 — CTR versus search position

Compare CTR across search-position buckets. This helps identify whether pages with similar positions have different click-through performance.

In [ ]:
signal_2_table = analysis_df.groupby("position_bucket", observed=False).agg(
    n=(ctr_col, "count"),
    median_ctr=(ctr_col, "median"),
    mean_ctr=(ctr_col, "mean")
).reset_index()

signal_2_table["verdict"] = "MIXED"
signal_2_table["explanation"] = "CTR summary by search-position bucket; compare groups with adequate sample sizes."

valid_signal_2 = signal_2_table.dropna(subset=["median_ctr"])
if len(valid_signal_2) >= 2:
    first_ctr = valid_signal_2["median_ctr"].iloc[0]
    last_ctr = valid_signal_2["median_ctr"].iloc[-1]
    if last_ctr < first_ctr:
        verdict_2 = "CONFIRMED"
    elif last_ctr > first_ctr:
        verdict_2 = "OPPOSITE"
    else:
        verdict_2 = "MIXED"
else:
    verdict_2 = "FALSE"

signal_2_table["verdict"] = verdict_2
display(signal_2_table)

## 6. Define the baseline scoring rule

The baseline prioritizes stale pages that have meaningful impressions and a positive CTR gap relative to their search-position peers.

**Score:** CTR gap × log(1 + impressions)

**Action:** REFRESH

**Reason code:** STALE_VISIBLE_CTR_GAP

This is a transparent heuristic baseline, not a trained predictive model.

In [ ]:
baseline_df = analysis_df.copy()

peer_ctr = baseline_df.groupby("position_bucket", observed=False)[ctr_col].median()
baseline_df["peer_median_ctr"] = baseline_df["position_bucket"].map(peer_ctr)

baseline_df["ctr_gap"] = (
    baseline_df["peer_median_ctr"] - baseline_df[ctr_col]
).clip(lower=0)

baseline_df["baseline_score"] = (
    baseline_df["ctr_gap"] * np.log1p(baseline_df[impressions_col].clip(lower=0))
)

eligible_mask = (
    (baseline_df[staleness_col] >= 180) &
    (baseline_df[impressions_col] >= 300) &
    (baseline_df[position_col] > 0) &
    baseline_df["peer_median_ctr"].notna() &
    baseline_df[ctr_col].notna() &
    (baseline_df["ctr_gap"] > 0)
)

baseline_df["action_label"] = np.where(eligible_mask, "REFRESH", "MONITOR")
baseline_df["reason_code"] = np.where(
    eligible_mask,
    "STALE_VISIBLE_CTR_GAP",
    "NOT_SELECTED"
)

baseline_df.loc[~eligible_mask, "baseline_score"] = 0

print("Action counts:")
display(baseline_df["action_label"].value_counts())

print("Reason-code counts:")
display(baseline_df["reason_code"].value_counts())

## 7. Build ranked action queue

Rank the highest-scoring content opportunities and save the queue as a CSV output.

In [ ]:
queue_columns = [
    content_col,
    staleness_col,
    impressions_col,
    ctr_col,
    position_col,
    "position_bucket",
    "peer_median_ctr",
    "ctr_gap",
    "baseline_score",
    "action_label",
    "reason_code"
]

ranked_queue = baseline_df[queue_columns].copy()
ranked_queue = ranked_queue.sort_values(
    by="baseline_score", ascending=False
).reset_index(drop=True)
ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))

queue_path = OUTPUT_DIR / "baseline_action_score.csv"
ranked_queue.to_csv(queue_path, index=False)

print("Saved ranked queue:", queue_path)
display(ranked_queue.head(20))

## 8. Top-20 review

Review the highest-ranked opportunities. Confidence is based on the number of records in the corresponding position bucket, not on a calibrated probability.

In [ ]:
top_20 = ranked_queue[ranked_queue["action_label"] == "REFRESH"].head(20).copy()

bucket_counts = baseline_df["position_bucket"].value_counts()

def confidence_note(bucket):
    count = int(bucket_counts.get(bucket, 0))
    if count >= 30:
        return "HIGH"
    elif count >= 10:
        return "MEDIUM"
    return "LOW"

top_20["confidence_note"] = top_20["position_bucket"].apply(confidence_note)
top_20["why_selected"] = (
    "Stale content with impressions and CTR below the position-bucket median."
)
top_20["what_could_make_it_wrong"] = (
    "Peer CTR may not reflect query intent, seasonality, brand demand, or content quality."
)

review_path = OUTPUT_DIR / "w04_top20_review.csv"
top_20.to_csv(review_path, index=False)

print("Top-20 review saved:", review_path)
display(top_20)

## 9. Weak-pick analysis

Inspect the selected recommendations with the smallest positive scores. These are weaker picks and may be sensitive to the thresholds.

In [ ]:
weak_picks = ranked_queue[ranked_queue["action_label"] == "REFRESH"].tail(10)

print("Weakest selected recommendations:")
display(weak_picks)

print("Total selected:", int((ranked_queue["action_label"] == "REFRESH").sum()))
print("Total monitored:", int((ranked_queue["action_label"] == "MONITOR").sum()))

## 10. Leakage check

Scan column names for terms that may indicate future outcomes or target labels. This is only a screening step; verify the actual meaning of every feature against the data dictionary.

In [ ]:
leakage_terms = [
    "future", "outcome", "target", "label", "conversion_after",
    "post_period", "next_period", "actual_result"
]

possible_leakage_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in leakage_terms)
]

print("Columns requiring leakage review:")
print(possible_leakage_columns)

print("\nScoring input columns:")
print([staleness_col, impressions_col, ctr_col, position_col])

## 11. Save baseline metrics

In [ ]:
metrics = {
    "dataset_rows": int(len(df)),
    "dataset_columns": int(len(df.columns)),
    "selected_refresh_count": int((baseline_df["action_label"] == "REFRESH").sum()),
    "monitor_count": int((baseline_df["action_label"] == "MONITOR").sum()),
    "top_20_review_count": int(len(top_20)),
    "signal_1_flag_column": flag_col,
    "signal_1_verdict": str(signal_1_table["verdict"].iloc[0]) if len(signal_1_table) else "FALSE",
    "signal_2_verdict": str(signal_2_table["verdict"].iloc[0]) if len(signal_2_table) else "FALSE",
    "possible_leakage_columns": possible_leakage_columns,
    "scoring_rule": "ctr_gap * log1p(impressions)",
    "action_label": "REFRESH",
    "reason_code": "STALE_VISIBLE_CTR_GAP"
}

metrics_path = OUTPUT_DIR / "w04_baseline_metrics.json"
with open(metrics_path, "w") as file:
    json.dump(metrics, file, indent=4)

print("Metrics saved:", metrics_path)
print(json.dumps(metrics, indent=4))

## 12. Final self-check

Confirm that the notebook created the ranked queue, Top-20 review, and metrics output.

In [ ]:
assert len(ranked_queue) == len(df), "Ranked queue row count does not match dataset."
assert "baseline_score" in ranked_queue.columns
assert "action_label" in ranked_queue.columns
assert "reason_code" in ranked_queue.columns
assert queue_path.exists(), "Ranked queue CSV was not created."
assert review_path.exists(), "Top-20 review CSV was not created."
assert metrics_path.exists(), "Metrics JSON was not created."

print("All basic self-checks passed.")
print("Remember to verify column mappings, flag semantics, bucket thresholds, and CTR units before committing.")